# Iniciando o Spark

In [ ]:
# Iniciando Spark
from pyspark.sql
import SparkSession
from pyspark.sql
import functions as F
from datetime import datetime
import pytz, os

In [ ]:
spark = SparkSession.builder.appName("TELCO_RAW").getOrCreate()

# Funções

In [ ]:
# Função de log
def log(): return datetime.now().strftime('%Y-%m-%d %H:%M:%S')

In [ ]:
# Timestamps
agora = datetime.now(pytz.timezone('America/Sao_Paulo'))
dthproc = agora.strftime("%Y%m%d%H%M%S")

# Ingestão dos Dados

In [ ]:
raw_path = "s3://meu-bucket/raw/base_telco"


In [ ]:
# Leitura parquet
df_Raw_base_telco = spark.read.parquet(raw_path)
df_Raw_base_telco.createOrReplaceTempView("df_Raw_base_telco")

In [ ]:
print(log(), "Registros na camada raw:", df_Raw_base_telco.count())

# Carga na camada Raw

In [ ]:
# Criação da tabela lake a partir da raw
df_lake_Raw_base_telco = spark.sql(f"""
    SELECT
        INT(date_format(CREATED_AT,'yyyyMM')) AS ref,
        INT(date_format(CREATED_AT,'yyyyMM')) AS ref_partition,

        {dthproc} AS ts_proc,
        {dthproc} AS ts_proc_partition,

        -- todos os campos originais como STRING
        NUM_CPF,
        SAFRA,
        FLAG_INSTALACAO,
        FPD,
        PROD,
        flag_mig2,
        {",".join([f"var_{i}" for i in range(26,94)])},
        CREATED_AT
    FROM raw_base_telco
""")

# Registrar view temporária
df_lake_Raw_base_telco.createOrReplaceTempView("lake")

# Cache para otimizar consultas
df_lake_Raw_base_telco.cache()




In [ ]:
# Exibir contagem e schema
print("Registros no lake:", df_lake_Raw_base_telco.count())
df_lake_Raw_base_telco.printSchema()
df_lake_Raw_base_telco.show(5, truncate=False)

In [ ]:
# Registros de log
print(log(), "Registros no lake:", df_lake_Raw_base_telco.count())
df_lake_Raw_base_telco.printSchema()
df_lake_Raw_base_telco.show(5, truncate=False)

In [ ]:
# Escrita parquet particionado
bucket_raw = "s3://meu-bucket/0002_raw/base_telco"
df_lake_Raw_base_telco.write \
.partitionBy("ref_partition","ts_proc_partition") \
.mode("overwrite") \
.option("compression", "snappy") \
.parquet(bucket_raw)

In [ ]:
# Controle de carga
controle = spark.sql(f"""
   SELECT
        'base_telco' AS name_file,
        ref,
        ref_partition,
        ts_proc,
        ts_proc_partition,
        COUNT(*) AS qtd_registros
        FROM lake
        GROUP BY 1,2,3,4,5 """)
controle.createOrReplaceTempView("controle")
controle.cache()
print(log(), "Controle de carga:")


In [ ]:
controle.show(truncate=False)

In [ ]:
#Salvando os dados de controle
bucket_control = "s3://meu-bucket/0005_control/tb_0001_controle_procesamento_raw"
controle.write \
.mode("append") \
.option("compression", "snappy") \
.parquet(bucket_control)
